In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import urllib.request
import zipfile
import io

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# SECTION 2: LOAD DATASET

df = pd.read_csv("spam.csv", encoding='latin-1')

# Keep only required columns
df = df[['v1', 'v2']]

# Rename columns
df.columns = ['label', 'message']

print("First 5 Records:")
display(df.head())

print("\nDataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

In [ ]:
# SECTION 3: DATA EXPLORATION

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nSpam and Ham Count:")
print(df['label'].value_counts())

# Visualize class distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='label')
plt.title("Spam vs Ham Distribution")
plt.xlabel("Message Type")
plt.ylabel("Count")
plt.show()

In [ ]:
# SECTION 4: DATA PREPROCESSING

# Remove missing values
df = df.dropna()

# Remove duplicate messages
df = df.drop_duplicates()

# Convert labels into numerical values
# Ham = 0, Spam = 1
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

# Remove invalid labels if any
df = df.dropna(subset=['label'])

df['label'] = df['label'].astype(int)

print("\nDataset After Preprocessing:")
display(df.head())

print("Shape:", df.shape)

In [ ]:
# SECTION 5: FEATURE EXTRACTION USING TF-IDF

# Input messages
X = df['message']

# Target labels
y = df['label']

# Convert text into numerical features
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=3000
)

X_tfidf = tfidf.fit_transform(X)

print("\nTF-IDF Feature Shape:", X_tfidf.shape)

In [ ]:
# SECTION 6: SPLIT DATASET INTO TRAINING AND TESTING

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)


In [ ]:
# SECTION 7: K-NEAREST NEIGHBORS (KNN) CLASSIFIER

knn_model = KNeighborsClassifier(
    n_neighbors=5,
    metric='cosine',
    algorithm='brute'
)

# Train model
knn_model.fit(X_train, y_train)

# Predict results
knn_pred = knn_model.predict(X_test)

# Evaluate KNN
knn_accuracy = accuracy_score(y_test, knn_pred)
knn_precision = precision_score(y_test, knn_pred)
knn_recall = recall_score(y_test, knn_pred)
knn_f1 = f1_score(y_test, knn_pred)

print("\n========== KNN CLASSIFICATION RESULTS ==========")

print("Accuracy:", knn_accuracy)
print("Precision:", knn_precision)
print("Recall:", knn_recall)
print("F1 Score:", knn_f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    knn_pred,
    target_names=['Ham', 'Spam']
))

In [ ]:
# SECTION 8: SUPPORT VECTOR MACHINE (SVM) CLASSIFIER

svm_model = SVC(
    kernel='linear',
    random_state=42
)

# Train model
svm_model.fit(X_train, y_train)

# Predict results
svm_pred = svm_model.predict(X_test)

# Evaluate SVM
svm_accuracy = accuracy_score(y_test, svm_pred)
svm_precision = precision_score(y_test, svm_pred)
svm_recall = recall_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred)

print("\n========== SVM CLASSIFICATION RESULTS ==========")

print("Accuracy:", svm_accuracy)
print("Precision:", svm_precision)
print("Recall:", svm_recall)
print("F1 Score:", svm_f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    svm_pred,
    target_names=['Ham', 'Spam']
))

In [ ]:
# SECTION 9: CONFUSION MATRIX FOR KNN

knn_cm = confusion_matrix(y_test, knn_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    knn_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Ham', 'Spam'],
    yticklabels=['Ham', 'Spam']
)

plt.title("KNN Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()


In [ ]:
# SECTION 10: CONFUSION MATRIX FOR SVM

svm_cm = confusion_matrix(y_test, svm_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    svm_cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=['Ham', 'Spam'],
    yticklabels=['Ham', 'Spam']
)

plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()


In [ ]:
# SECTION 11: PERFORMANCE COMPARISON

results = pd.DataFrame({
    'Classifier': ['KNN', 'SVM'],
    'Accuracy': [knn_accuracy, svm_accuracy],
    'Precision': [knn_precision, svm_precision],
    'Recall': [knn_recall, svm_recall],
    'F1 Score': [knn_f1, svm_f1]
})

print("\n========== MODEL PERFORMANCE COMPARISON ==========")
display(results)

In [ ]:
# SECTION 12: VISUALIZE MODEL COMPARISON

results_plot = results.set_index('Classifier')

results_plot.plot(
    kind='bar',
    figsize=(10, 6)
)

plt.title("KNN vs SVM Performance Comparison")
plt.ylabel("Score")
plt.xlabel("Classifier")
plt.xticks(rotation=0)
plt.ylim(0, 1.1)
plt.legend(loc='lower right')
plt.grid(axis='y')
plt.show()

In [ ]:
# SECTION 13: FINAL CONCLUSION

print("\n========== FINAL CONCLUSION ==========")

if svm_accuracy > knn_accuracy:
    print("SVM performs better than KNN in terms of accuracy.")
elif knn_accuracy > svm_accuracy:
    print("KNN performs better than SVM in terms of accuracy.")
else:
    print("Both classifiers have equal accuracy.")

if svm_f1 > knn_f1:
    print("SVM has a better F1 Score.")
elif knn_f1 > svm_f1:
    print("KNN has a better F1 Score.")
else:
    print("Both classifiers have equal F1 Scores.")

print("\nFinal Accuracy:")
print("KNN Accuracy:", round(knn_accuracy * 100, 2), "%")
print("SVM Accuracy:", round(svm_accuracy * 100, 2), "%")

print("\nPractical Completed Successfully!")
